# Customer Pilot Engineering

Turn an ambiguous customer request into an accepted, observable pilot with an operator handoff.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Discover and integrate


### Lecture 01 · Discovery and pilot acceptance

Translate a request into a workflow, baseline, and measurable acceptance criteria.

Use the baseline time measurements in the workbook. Write a target and a guardrail that would both be observable during a pilot.


In [ ]:
from statistics import median

baseline_minutes = (10, 11, 12, 13, 15)
median_target = median(baseline_minutes) * 0.75
assert median_target == 9
print("Median baseline and acceptance target", median(baseline_minutes), median_target)


**Compare your result**

A defensible starting contract targets a 25% reduction in median time and requires every restricted case to be handed off.


### Lecture 02 · Map a customer data contract

Normalize a source payload and publish a versioned integration boundary.

Run the adapter for the bundled records. Add a case for a field with the wrong type.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Ticket:
    source_id: str
    topic: str
    restricted: bool


def normalize(payload):
    if payload.get("schema") != "facilities-v1":
        raise ValueError("unsupported schema")
    source = payload.get("ticket")
    topic = payload.get("queue")
    restricted = payload.get("restricted")
    if not isinstance(source, str) or not source.strip() or not isinstance(topic, str) or type(restricted) is not bool:
        raise ValueError("invalid ticket fields")
    return Ticket(source.strip(), topic.strip().lower(), restricted)


inputs = [
    {"schema": "facilities-v1", "ticket": "F1", "queue": " Heating ", "restricted": False},
    {"schema": "facilities-v1", "ticket": "F2", "queue": "heating", "restricted": True},
    {"schema": "facilities-v1", "ticket": "F3", "queue": "other", "restricted": False},
]
tickets = [normalize(row) for row in inputs]
assert tickets[0] == Ticket("F1", "heating", False)
try:
    normalize(inputs[0] | {"restricted": "false"})
except ValueError as error:
    print("Rejected malformed request", error)
else:
    raise AssertionError("wrong field type was accepted")


**Compare your result**

Treat the malformed input as a rejected record. Leave the original source identifier available for support.


## Evaluate and hand off


### Lecture 03 · Shadow acceptance and integration failures

Exercise normal, ambiguous, restricted, and unavailable-dependency cases before live use.

Compare the deterministic acceptance results. Add a replay for a successful request and a retry after dependency recovery.


In [ ]:
class Intake:
    def __init__(self):
        self.records = {}

    def shadow(self, ticket, lookup):
        saved = self.records.get(ticket.source_id)
        if saved:
            if saved[0] != ticket:
                raise ValueError("source ID reused with different inputs")
            return saved[1]
        try:
            route = lookup(ticket.topic) if not ticket.restricted else None
        except TimeoutError:
            return "retry_manual"
        result = route or "review"
        self.records[ticket.source_id] = (ticket, result)
        return result


def lookup(topic):
    return {"heating": "building-services"}.get(topic)


def unavailable(topic):
    raise TimeoutError("directory did not respond")


intake = Intake()
assert intake.shadow(tickets[0], unavailable) == "retry_manual"
assert "F1" not in intake.records
assert [intake.shadow(ticket, lookup) for ticket in tickets] == ["building-services", "review", "review"]
assert intake.shadow(tickets[0], unavailable) == "building-services"
assert len(intake.records) == 3


**Compare your result**

The successful replay returns the same result and creates one record. A transient failure leaves room for a successful retry.


### Lecture 04 · Rollout, operator handoff and stop rules

Write a release recommendation grounded in measured outcomes and assign an operational owner.

Review both bundled pilot result sets. Write a one-page handoff with the same triggers.


In [ ]:
def decide(observed_times, restricted_case_ok, completed_smoke_test):
    return (
        len(observed_times) >= len(baseline_minutes)
        and median(observed_times) <= median_target
        and restricted_case_ok
        and completed_smoke_test
    )


pilot_minutes = (6, 7, 8, 9, 11)
assert decide(pilot_minutes, True, True)
assert not decide(pilot_minutes, False, True)
assert not decide(pilot_minutes[:2], True, True)
print({
    "pilot_cohort": "simulated facilities intake",
    "cases": len(pilot_minutes),
    "median_minutes": median(pilot_minutes),
    "recommendation": "bounded pilot",
    "operator_owner": "facilities shift lead",
    "stop_rule": "route any restricted case manually and pause automation",
    "smoke_case": {"input_id": "F2", "expected": "review"},
})


**Compare your result**

Recommend controlled adoption only when the thresholds pass. Identify who can pause the service and how affected work is handled.


## Course project

Deliver a pilot for a simulated facilities team. Include the workflow baseline, an adapter, a replay-safe intake path, acceptance evidence, and a handoff.

Show an accepted request, unverifiable evidence, a temporary dependency failure, and a replay. Record the scope, pilot owner, stop rule, and what the customer needs to operate it.
